In [10]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor

from utils.config import load_config
from preprocessing.house_prices_preprocessing import HousePricesPreprocessor

In [11]:
# --- config ---
config = load_config()

# --- загрузка ---
train = pd.read_csv(config["paths"]["train"])
test = pd.read_csv(config["paths"]["test"])

print(train.shape, test.shape)

(1460, 81) (1459, 80)


In [12]:
# --- target ---
target = "SalePrice"

y = np.log1p(train[target])   # логарифмируем
X = train.drop(columns=[target])

# --- preprocessing ---
preprocessor = HousePricesPreprocessor(outlier_quantile=0.95)

X_prepared = preprocessor.fit_transform(X)
X_test_prepared = preprocessor.transform(test)

print(X_prepared.shape, X_test_prepared.shape)

(1460, 178) (1459, 178)


In [13]:
best_params = {
    "iterations": 920,
    "learning_rate": 0.06152055348604786,
    "depth": 4,
    "l2_leaf_reg": 7,
    "random_strength": 3,
    "subsample": 0.8,

    "loss_function": "RMSE",
    "verbose": 0,
    "random_state": config["seed"],
    "bootstrap_type": "Bernoulli",
}

model = CatBoostRegressor(**best_params)

# обучаемся на ВСЕМ train
model.fit(X_prepared, y)

CatBoostRegressor(bootstrap_type='Bernoulli', depth=4, iterations=920, l2_leaf_reg=7, learning_rate=0.06152055348604786, loss_function='RMSE', random_state=42, random_strength=3, subsample=0.8, verbose=0)

In [14]:
# --- предсказание в лог-пространстве ---
y_pred_log = model.predict(X_test_prepared)

# --- возвращаем в реальные значения ---
y_pred = np.expm1(y_pred_log)

In [15]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": y_pred
})

submission.head()

,Id,SalePrice
0,1461,129669.750879
1,1462,161765.195500
2,1463,177677.331002
3,1464,189907.419601
4,1465,183352.637373


In [16]:
print(submission["SalePrice"].describe())

# нет ли отрицательных
print("Есть отрицательные:", (submission["SalePrice"] < 0).any())

count      1459.000000
mean     177004.461262
std       74684.768841
min       38850.208232
25%      127781.178109
50%      155945.630418
75%      207723.197137
max      512448.301883
Name: SalePrice, dtype: float64
Есть отрицательные: False


In [17]:
submission.to_csv("../../submission/submission.csv", index=False)